# Exercise — Interpret Quality Results & Plan Remediation

Run **Trailhead Provisions**' quality suite (Great-Expectations-shaped results across six
dimensions). For each failure, choose a remediation strategy — **source**, **downstream**, or
**quarantine** — and write a prioritized plan tied to the platform's target contract SLOs. See `INSTRUCTIONS.md`.

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your AWS Cloud Lab environment and seed the AWS Glue Data Catalog (`trailhead`) if it is not yet present in your lab account.

In [ ]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

# Auto-load .env from the workspace or parent directories if present
from pathlib import Path
for _parent in (Path.cwd(), *Path.cwd().parents):
    _env_file = _parent / ".env"
    if _env_file.is_file():
        for _raw in _env_file.read_text(encoding="utf-8").splitlines():
            _line = _raw.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                _k, _v = _k.strip(), _v.strip().strip("\"").strip("'")
                if _k and _v and not os.environ.get(_k):
                    os.environ[_k] = _v
        break

import boto3
from botocore.exceptions import ClientError, NoCredentialsError
import setup_aws

_CRED_MSG = (
    "AWS credentials are missing or expired.\n"
    "1. Open Cloud Resources and start your Cloud Lab.\n"
    "2. Copy AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, and AWS_SESSION_TOKEN.\n"
    "3. Uncomment and paste them into Step 0 above, then re-run Step 0 (and re-run in each new lab session)."
)

# Verify credentials before seeding
sts = boto3.client("sts", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    sts.get_caller_identity()
except NoCredentialsError:
    raise RuntimeError(_CRED_MSG) from None
except ClientError as exc:
    code = exc.response.get("Error", {}).get("Code", "")
    if code in {"ExpiredToken", "ExpiredTokenException", "InvalidClientTokenId", "UnrecognizedClientException", "AuthFailure", "AccessDeniedException"}:
        raise RuntimeError(_CRED_MSG) from None
    raise
# Seed the Glue Data Catalog, S3 tables, and Lake Formation enforcement if needed
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")
q = gc.run_quality_suite()
print("Failing expectations:", int((~q["success"]).sum()), "of", len(q))
q

In [ ]:
display(q.groupby("dimension").agg(checks=("success", "size"),
                                   failing=("success", lambda s: int((~s).sum())),
                                   worst_pct=("unexpected_percent", "max")).reset_index())
print("--- Target Contract SLOs ---")
gc.contract_slos()

## 1. Classify each failing expectation
Map every `table.column` failure to `source` | `downstream` | `quarantine`.

In [ ]:
# TODO(you): map each failing table.column to "source" | "downstream" | "quarantine".
remediation = {
    # "product.category": "source",
}
fails = q[~q["success"]].assign(key=lambda d: d["table"] + "." + d["column"])
fails["strategy"] = fails["key"].map(remediation)
fails[["key", "dimension", "unexpected_percent", "strategy"]]

## 2. Prioritized remediation plan
Replace the cell below with your plan.

> **Your remediation plan here.** Address:
>
> - A strategy (`source` / `downstream` / `quarantine`) per issue, with justification.
> - Prioritization: which failures cause active GDPR or CSRD regulatory harm and must be quarantined first?
> - Tie every remediation tier to the target contract SLOs from `gc.contract_slos()`.
> - Which issues are recurring drift (consent, carbon unit) and need ongoing SLOs.